In [7]:
"""
correlate_tau_vs_cost.py
=========================
Correlates the 11 sampled tau (time-constant) parameters against the
optimizer's cost/fitness function ("score" column) from
LI_optimisation_results.csv.

Why log2-ratio and not raw tau:
  The exploration script samples each tau UNIFORMLY in log2(tau/baseline)
  space (see TIME_PARAM_KEYS / get_max_bound in the LHS script), not in
  raw physical units. Correlating raw tau values would be dominated by
  each parameter's arbitrary physical scale (e.g. tau_r_W ~ 1e2 vs
  tau_c_e_GS ~ 1e-12) and by the multiplicative nature of the sampling.
  Converting each tau to log2(tau / baseline) puts every parameter on the
  same dimensionless footing the optimizer actually explores.

Why the correlation is split into "valid-only" and "all rows":
  score == -RMSE only for runs that completed and produced a usable L-I
  curve. Crashed / incomplete-sim / dead-laser runs instead get a large
  FIXED penalty (-1e6 / -5e4 / -1e4) unrelated to curve-fit quality. Left
  in an all-rows Pearson correlation, those penalty outliers (up to -1e6
  vs a valid-run range of roughly -2.4 to -0.2) completely dominate the
  covariance and swamp any real RMSE-vs-tau relationship. So:
    - "valid-only" correlation = how each tau relates to fit quality
      (RMSE) among runs that actually produced a curve.
    - "failure association" = a separate, simple check for whether large
      offsets in a given tau tend to coincide with crashes/failures.

Outputs:
  - Printed Pearson & Spearman correlation table (valid-only)
  - Printed failure-association summary (all rows)
  - tau_score_correlation.png (bar chart, valid-only Pearson r)
"""

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr, pointbiserialr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV_PATH = r"C:\Users\josep\Documents\MRes mini-project 2\Figures\Correlations\water_87ck.csv"
OUT_PNG = "tau_score_correlation.png"

# Baselines and bounds copied from the LHS exploration script, needed to
# convert each raw tau value back into the log2(ratio) space it was
# actually sampled in.
TAU_BASELINE = {
    "tau_aug_GS": 660e-12, "tau_aug_ES1": 275e-12, "tau_aug_ES2": 110e-12,
    "tau_spon_GS": 0.1 * 2.8e-9, "tau_spon_ES1": 0.1 * 2.8e-9, "tau_spon_ES2": 0.1 * 2.8e-9,
    "tau_c_e_GS": 2e-12, "tau_c_e_ES1": 3e-12, "tau_c_e_ES2": 3e-12,
    "tau_c_e_W": 1.2, "tau_r_W": 100,
}
MAX_TIME_MULT = 10.0
MAX_TIME_LOG2 = np.log2(MAX_TIME_MULT)

# LaTeX (mathtext) symbols for each tau parameter, used for axis tick
# labels instead of the raw variable names.
TAU_LATEX = {
    "tau_aug_GS":   r"$\tau_{aug}^{GS}$",
    "tau_aug_ES1":  r"$\tau_{aug}^{ES1}$",
    "tau_aug_ES2":  r"$\tau_{aug}^{ES2}$",
    "tau_spon_GS":  r"$\tau_{spon}^{GS}$",
    "tau_spon_ES1": r"$\tau_{spon}^{ES1}$",
    "tau_spon_ES2": r"$\tau_{spon}^{ES2}$",
    "tau_c_e_GS":   r"$\tau_{c,e}^{GS}$",
    "tau_c_e_ES1":  r"$\tau_{c,e}^{ES1}$",
    "tau_c_e_ES2":  r"$\tau_{c,e}^{ES2}$",
    "tau_c_e_W":    r"$\tau_{c,e}^{W}$",
    "tau_r_W":      r"$\tau_{r}^{W}$",
}

# ── Figure sizing (golden ratio, single-column width) ──────────────────────
GOLDEN_RATIO = 1.618
FIG_WIDTH_IN = 4
FIG_HEIGHT_IN = FIG_WIDTH_IN / GOLDEN_RATIO

# ── Publication-style rcParams ──────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,   # embed fonts properly (editable text in PDF)
    "ps.fonttype": 42,
})


def to_log2_ratio(series, baseline):
    ratio = series.astype(float) / baseline
    ratio = ratio.clip(lower=1e-12)
    return np.clip(np.log2(ratio), -MAX_TIME_LOG2, MAX_TIME_LOG2)


def main():
    df = pd.read_csv(CSV_PATH)
    tau_keys = list(TAU_BASELINE.keys())

    # Log2-ratio version of every tau column, on the same footing the
    # sampler actually used.
    for k in tau_keys:
        df[f"log2_{k}"] = to_log2_ratio(df[k], TAU_BASELINE[k])

    valid = df[df["invalid"] == False].copy()
    print(f"Total rows: {len(df)}  |  Valid (curve-fit) rows: {len(valid)}  "
          f"|  Invalid/failed rows: {len(df) - len(valid)}\n")

    # --- 1. Valid-only correlation: log2(tau ratio) vs score (=-RMSE) ---
    rows = []
    for k in tau_keys:
        x = valid[f"log2_{k}"].values
        y = valid["score"].values
        pr, pp = pearsonr(x, y)
        sr, sp = spearmanr(x, y)
        rows.append({
            "tau_param": k,
            "pearson_r": pr, "pearson_p": pp,
            "spearman_r": sr, "spearman_p": sp,
        })
    corr_df = pd.DataFrame(rows).sort_values("pearson_r", key=np.abs, ascending=False)
    corr_df = corr_df.reset_index(drop=True)

    pd.set_option("display.width", 120)
    print("=" * 78)
    print("VALID-ONLY correlation: log2(tau / baseline)  vs  score (= -RMSE)")
    print("(sorted by |Pearson r|; score closer to 0 = better fit)")
    print("=" * 78)
    print(corr_df.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

    # --- 2. Failure association (all rows): does a large tau offset
    #        coincide with crash/incomplete/dead-laser more than a
    #        smaller offset does? Point-biserial correlation between
    #        |log2 ratio| and the binary "invalid" flag.
    fail_rows = []
    for k in tau_keys:
        x = df[f"log2_{k}"].abs().values
        y = df["invalid"].astype(int).values
        r, p = pointbiserialr(y, x)
        fail_rows.append({"tau_param": k, "pointbiserial_r_vs_failure": r, "p_value": p})
    fail_df = pd.DataFrame(fail_rows).sort_values(
        "pointbiserial_r_vs_failure", key=np.abs, ascending=False).reset_index(drop=True)

    print()
    print("=" * 78)
    print("ALL-ROWS check: |log2(tau / baseline)|  vs  failure (crash/incomplete/dead)")
    print("(positive r => bigger offset on this axis => more likely to fail)")
    print("=" * 78)
    print(fail_df.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

    # --- Plot: valid-only Pearson r per tau parameter ---
    plot_df = corr_df.sort_values("pearson_r")
    labels = [TAU_LATEX[k] for k in plot_df["tau_param"]]
    colors = ["#c0392b" if v < 0 else "#2471a3" for v in plot_df["pearson_r"]]

    fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))
    ax.barh(labels, plot_df["pearson_r"], color=colors, height=0.65)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel(r"Pearson $r$  ($\tau_x$ vs. score)")
    ax.grid(True, axis="x", linestyle=":", alpha=0.6)
    fig.tight_layout()
    fig.savefig(r"C:\Users\josep\Documents\MRes mini-project 2\Figures\Ultra-stable-lasers-and-photonic-integrated-circuits-for-secure-communications\content\Figures\Methods & Results\pearson_correlation.png", dpi=300)
    print(f"\nSaved chart: {OUT_PNG}")


if __name__ == "__main__":
    main()

Total rows: 1763  |  Valid (curve-fit) rows: 1636  |  Invalid/failed rows: 127

VALID-ONLY correlation: log2(tau / baseline)  vs  score (= -RMSE)
(sorted by |Pearson r|; score closer to 0 = better fit)
   tau_param  pearson_r  pearson_p  spearman_r  spearman_p
  tau_aug_GS     0.5754     0.0000      0.5778      0.0000
     tau_r_W     0.3833     0.0000      0.2491      0.0000
 tau_spon_GS     0.1824     0.0000      0.1549      0.0000
tau_spon_ES1     0.1300     0.0000      0.1207      0.0000
 tau_c_e_ES2    -0.0702     0.0045      0.0026      0.9176
 tau_aug_ES1     0.0478     0.0533      0.0417      0.0915
tau_spon_ES2    -0.0326     0.1874     -0.0483      0.0506
   tau_c_e_W    -0.0268     0.2782     -0.0285      0.2500
  tau_c_e_GS     0.0159     0.5198      0.0311      0.2083
 tau_c_e_ES1    -0.0125     0.6121      0.0036      0.8838
 tau_aug_ES2    -0.0059     0.8108     -0.0059      0.8112

ALL-ROWS check: |log2(tau / baseline)|  vs  failure (crash/incomplete/dead)
(positive r =

In [10]:
"""
correlate_tau_vs_occupation_cost.py
=====================================
Builds a cost function from the final occupation states -- the RMS
deviation of (GS, ES1, ES2) at 50 mA from the legacy population targets
(GS=0.20, ES1=0.40, ES2=0.50):

    occupation_cost = sqrt( mean( (GS-0.20)^2, (ES1-0.40)^2, (ES2-0.50)^2 ) )

-- and correlates each of the 11 sampled tau (time-constant) parameters
against a SCORE built from it, mirroring the earlier tau-vs-RMSE
correlation analysis but with the L-I curve-fit RMSE swapped out for
this occupation-target cost.

Same log2-ratio treatment as before: each tau is converted to
log2(tau/baseline) since that's the space the LHS sampler actually draws
from, not raw physical units.

SIGN CONVENTION (flipped from the previous version of this script):
occupation_cost itself is >= 0, with 0 meaning a perfect match to the
population targets -- bigger is worse. To correlate in the same
"positive r = improves" sense as the original tau-vs-fit-quality chart
(which plotted tau against score=-RMSE, not RMSE directly), this version
correlates tau against occupation_score = -occupation_cost instead:
  POSITIVE r = increasing that tau moves the occupation state CLOSER to
               target (improves the match, occupation_score -> 0)
  NEGATIVE r = increasing that tau moves the occupation state FURTHER
               from target (worsens the match)
This is called out explicitly in the chart title, and bar colour now
matches: blue = positive = improves, red = negative = worsens.

Output: tau_vs_occupation_cost_correlation.png
"""

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV_PATH = r"C:\Users\josep\Documents\MRes mini-project 2\Figures\Correlations\water_87ck.csv"
OUT_PNG = "tau_vs_occupation_cost_correlation.png"

TARGETS = {"GS": 0.20, "ES1": 0.40, "ES2": 0.50}
OCC_COLS = {"GS": "occupation_GS_50mA", "ES1": "occupation_ES1_50mA",
            "ES2": "occupation_ES2_50mA"}

TAU_BASELINE = {
    "tau_aug_GS": 660e-12, "tau_aug_ES1": 275e-12, "tau_aug_ES2": 110e-12,
    "tau_spon_GS": 0.1 * 2.8e-9, "tau_spon_ES1": 0.1 * 2.8e-9, "tau_spon_ES2": 0.1 * 2.8e-9,
    "tau_c_e_GS": 2e-12, "tau_c_e_ES1": 3e-12, "tau_c_e_ES2": 3e-12,
    "tau_c_e_W": 1.2, "tau_r_W": 100,
}
MAX_TIME_MULT = 10.0
MAX_TIME_LOG2 = np.log2(MAX_TIME_MULT)


def to_log2_ratio(series, baseline):
    ratio = series.astype(float) / baseline
    ratio = ratio.clip(lower=1e-12)
    return np.clip(np.log2(ratio), -MAX_TIME_LOG2, MAX_TIME_LOG2)


def main():
    df = pd.read_csv(CSV_PATH)

    # Coalesce old/new schema columns (see earlier column-rename note)
    df["occupation_GS_50mA"] = df["occupation_GS_50mA"].fillna(df["final_gs"])
    df["occupation_ES1_50mA"] = df["occupation_ES1_50mA"].fillna(df["final_es1"])
    df["occupation_ES2_50mA"] = df["occupation_ES2_50mA"].fillna(df["final_es2"])

    valid = df[df["invalid"] == False].copy()
    print(f"Valid rows: {len(valid)}")

    # --- Cost function: RMS deviation from population targets (>=0, bigger=worse) ---
    sq_devs = sum((valid[OCC_COLS[k]] - TARGETS[k]) ** 2 for k in TARGETS)
    valid["occupation_cost"] = np.sqrt(sq_devs / 3.0)
    print(valid["occupation_cost"].describe())
    print()

    # --- Score used for correlation: flip sign so positive = improves ---
    valid["occupation_score"] = -valid["occupation_cost"]

    # --- log2-ratio version of every tau column ---
    tau_keys = list(TAU_BASELINE.keys())
    for k in tau_keys:
        valid[f"log2_{k}"] = to_log2_ratio(valid[k], TAU_BASELINE[k])

    rows = []
    for k in tau_keys:
        x = valid[f"log2_{k}"].values
        y = valid["occupation_score"].values
        pr, pp = pearsonr(x, y)
        sr, sp = spearmanr(x, y)
        rows.append({"tau_param": k, "pearson_r": pr, "pearson_p": pp,
                      "spearman_r": sr, "spearman_p": sp})
    corr_df = pd.DataFrame(rows).sort_values("pearson_r", key=np.abs, ascending=False)
    corr_df = corr_df.reset_index(drop=True)

    pd.set_option("display.width", 120)
    print("=" * 78)
    print("VALID-ONLY correlation: log2(tau / baseline)  vs  occupation_score (=-occupation_cost)")
    print("(sorted by |Pearson r|; occupation_score = 0 means a perfect match")
    print(" to GS=0.20/ES1=0.40/ES2=0.50; POSITIVE r = this tau moves the")
    print(" occupation state CLOSER to target (improves); NEGATIVE r = FURTHER")
    print(" from target (worsens))")
    print("=" * 78)
    print(corr_df.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

    # --- Plot ---
    plot_df = corr_df.sort_values("pearson_r")
    # blue = positive = improves, red = negative = worsens (matches the
    # original tau-vs-fit-quality chart's colour convention)
    colors = ["#2471a3" if v > 0 else "#c0392b" for v in plot_df["pearson_r"]]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(plot_df["tau_param"], plot_df["pearson_r"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Pearson r  (log2(tau/baseline)  vs  occupation_score)")
    ax.set_title("Correlation of tau parameters with occupation-target score\n"
                  "(valid runs only)\n"
                  "occupation_score = -sqrt(mean((GS-0.20)\u00b2,(ES1-0.40)\u00b2,(ES2-0.50)\u00b2));\n"
                  "positive r = increasing this tau IMPROVES the occupation-target match")
    ax.grid(True, axis="x", linestyle=":", alpha=0.6)
    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=150)
    print(f"\nSaved chart: {OUT_PNG}")

    valid.to_csv("valid_with_occupation_cost.csv", index=False)


if __name__ == "__main__":
    main()

Valid rows: 1636
count    1636.000000
mean        0.170644
std         0.005778
min         0.151090
25%         0.167391
50%         0.171385
75%         0.174796
max         0.182431
Name: occupation_cost, dtype: float64

VALID-ONLY correlation: log2(tau / baseline)  vs  occupation_score (=-occupation_cost)
(sorted by |Pearson r|; occupation_score = 0 means a perfect match
 to GS=0.20/ES1=0.40/ES2=0.50; POSITIVE r = this tau moves the
 occupation state CLOSER to target (improves); NEGATIVE r = FURTHER
 from target (worsens))
   tau_param  pearson_r  pearson_p  spearman_r  spearman_p
 tau_c_e_ES1     0.9105     0.0000      0.9189      0.0000
  tau_c_e_GS     0.3195     0.0000      0.3194      0.0000
     tau_r_W     0.1042     0.0000      0.1026      0.0000
 tau_aug_ES2     0.0507     0.0403      0.0616      0.0127
   tau_c_e_W     0.0404     0.1023      0.0324      0.1908
 tau_c_e_ES2    -0.0377     0.1272     -0.0428      0.0836
 tau_aug_ES1     0.0152     0.5386      0.0157      0.